# Experimental Submission Notebook

This notebook is an experimental standalone version of our Kaggle submission pipeline.  
It is intended to make the submission workflow easy to run and inspect in one place, without relying on the local `src/` package structure.

A more complete description of the methodology, project structure, implementation details, and design choices is provided in the repository.

In [4]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from __future__ import annotations

import importlib


def ensure_runtime_dependencies() -> None:
    required_modules = {
        "numpy": "numpy",
        "pandas": "pandas",
        "sklearn": "scikit-learn",
        "rank_bm25": "rank-bm25",
        "sentence_transformers": "sentence-transformers",
        "torch": "torch",
    }
    missing_packages: list[str] = []
    for module_name, package_name in required_modules.items():
        try:
            importlib.import_module(module_name)
        except ImportError:
            missing_packages.append(package_name)
    if missing_packages:
        packages = " ".join(missing_packages)
        raise ImportError(
            f"Missing packages: {missing_packages}. Install them before running this notebook, for example: %pip install {packages}"
        )


ensure_runtime_dependencies()


# --- Begin src/types.py ---


from dataclasses import dataclass
from pathlib import Path
from typing import Literal, TypedDict

ModelName = Literal["tfidf", "bm25", "embedding"]


class RetrievalResult(TypedDict):
    query_id: str
    relevant_docs: list[str]


class GroundTruthEntry(TypedDict):
    relevant_doc_ids: set[str]
    total_relevant_docs: int
    category: str | None


@dataclass(frozen=True)
class RuntimePaths:
    runtime_env: Literal["colab", "kaggle", "local"]
    project_dir: Path
    work_dir: Path
    data_dir: Path
    cache_dir: Path
    output_path: Path


# --- End src/types.py ---


# --- Begin src/config.py ---


from dataclasses import dataclass, field



@dataclass(frozen=True)
class NormalizationConfig:
    lowercase: bool = True
    replace_separators: bool = True
    separator_chars: str = "-_/"
    collapse_whitespace: bool = True
    strip: bool = True


@dataclass(frozen=True)
class TFIDFConfig:
    lowercase: bool = True
    ngram_range: tuple[int, int] = (1, 2)
    min_df: int = 2


@dataclass(frozen=True)
class BM25Config:
    k1: float = 1.5
    b: float = 0.75
    delta: float = 1.0


@dataclass(frozen=True)
class EmbeddingConfig:
    model_name: str = "all-MiniLM-L6-v2"
    batch_size: int = 256
    query_chunk_size: int = 32


@dataclass(frozen=True)
class CrossEncoderConfig:
    model_name: str = "cross-encoder/ms-marco-MiniLM-L6-v2"
    epochs: int = 5
    batch_size: int = 32
    infer_batch_size: int = 64
    max_length: int = 256
    max_positives_per_query: int = 4
    negatives_per_positive: int = 4
    hard_negative_top_k: int = 200
    train_query_limit: int = 327
    rerank_top_m: int = 45
    zscore_weight: float = 1.0
    category_bonus: float = 0.5
    fp16: bool = True
    random_seed: int = 42
    enable_cache: bool = True


@dataclass(frozen=True)
class RetrievalPipelineConfig:
    final_model: ModelName = "embedding"
    evaluation_models: tuple[ModelName, ...] = ("embedding",)
    evaluation_top_ks: tuple[int, ...] = (12_500,)
    submit_top_k: int = 12_500
    enable_category_prediction: bool = True
    enable_category_filter: bool = True
    enable_cross_encoder_rerank: bool = True
    enable_embedding_cache: bool = True
    enable_classic_cache: bool = True
    token_pattern: str = r"[a-z0-9]+"


@dataclass(frozen=True)
class DataColumns:
    document_text_columns: tuple[str, ...] = ("title", "text", "tags")
    retrieval_query_columns: tuple[str, ...] = ("title", "text")
    classifier_query_columns: tuple[str, ...] = ("title", "text", "tags")
    use_query_tags_in_classifier: bool = True


@dataclass(frozen=True)
class DiagnosticsConfig:
    top_category_stats_k: int = 20
    query_category_stats_limit: int = 5


@dataclass(frozen=True)
class AllConfig:
    normalization: NormalizationConfig = field(default_factory=NormalizationConfig)
    tfidf: TFIDFConfig = field(default_factory=TFIDFConfig)
    bm25: BM25Config = field(default_factory=BM25Config)
    embedding: EmbeddingConfig = field(default_factory=EmbeddingConfig)
    cross_encoder: CrossEncoderConfig = field(default_factory=CrossEncoderConfig)
    retrieval_pipeline: RetrievalPipelineConfig = field(default_factory=RetrievalPipelineConfig)
    data_columns: DataColumns = field(default_factory=DataColumns)
    diagnostics: DiagnosticsConfig = field(default_factory=DiagnosticsConfig)


DEFAULT_CONFIG = AllConfig()


# --- End src/config.py ---


# --- Begin src/cache/store.py ---


import hashlib
import json
import pickle
import re
from pathlib import Path
from typing import Any, Sequence

import pandas as pd


MODEL_MEMORY_CACHE: dict[str, Any] = {}
ARRAY_MEMORY_CACHE: dict[str, Any] = {}
OBJECT_MEMORY_CACHE: dict[str, Any] = {}


def ensure_cache_dirs(cache_dir: Path) -> dict[str, Path]:
    directories = {
        "root": cache_dir,
        "sentence_transformers": cache_dir / "sentence_transformers",
        "embeddings": cache_dir / "embeddings",
        "tfidf": cache_dir / "tfidf",
        "bm25": cache_dir / "bm25",
        "classifier": cache_dir / "classifier",
        "cross_encoder": cache_dir / "cross_encoder",
    }
    for path in directories.values():
        path.mkdir(parents=True, exist_ok=True)
    return directories


def _safe_component(value: Any) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value))


def _hash_payload(payload: dict[str, Any]) -> str:
    raw_payload = json.dumps(payload, sort_keys=True, ensure_ascii=True, default=str)
    return hashlib.sha1(raw_payload.encode("utf-8")).hexdigest()[:16]


def _normalization_signature(config: AllConfig = DEFAULT_CONFIG) -> str:
    payload = {
        "normalization": config.normalization.__dict__,
        "token_pattern": config.retrieval_pipeline.token_pattern,
    }
    return _hash_payload(payload)


def _dataframe_fingerprint(frame: pd.DataFrame, columns: Sequence[str]) -> str:
    hasher = hashlib.sha1()
    hasher.update(str(len(frame)).encode("utf-8"))
    for column in columns:
        hasher.update(column.encode("utf-8"))
        column_hash = pd.util.hash_pandas_object(frame[column].astype(str), index=False).values
        hasher.update(column_hash.tobytes())
    return hasher.hexdigest()[:16]


def _load_pickle(path: Path) -> Any:
    with open(path, "rb") as handle:
        return pickle.load(handle)


def _save_pickle(path: Path, artifact: Any) -> None:
    with open(path, "wb") as handle:
        pickle.dump(artifact, handle, protocol=pickle.HIGHEST_PROTOCOL)


# --- End src/cache/store.py ---


# --- Begin src/data/loading.py ---


from pathlib import Path
from typing import Sequence

import pandas as pd


def require_columns(frame: pd.DataFrame, required_columns: Sequence[str], frame_name: str) -> None:
    missing_columns = [column for column in required_columns if column not in frame.columns]
    if missing_columns:
        raise ValueError(f"{frame_name} is missing required columns: {missing_columns}")


def ensure_unique_ids(frame: pd.DataFrame, frame_name: str) -> None:
    require_columns(frame, ["id"], frame_name)
    if not frame["id"].astype(str).is_unique:
        raise ValueError(f"{frame_name} contains duplicate ids, which would break retrieval output mapping.")


def load_json_frame(path: Path, frame_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{frame_name} file not found: {path}")
    return pd.read_json(path)


# --- End src/data/loading.py ---


# --- Begin src/data/text.py ---


import re
from typing import Any, Sequence

import pandas as pd


_TOKEN_RE = re.compile(DEFAULT_CONFIG.retrieval_pipeline.token_pattern)
_WHITESPACE_RE = re.compile(r"\s+")
_SEPARATOR_RE = re.compile(f"[{re.escape(DEFAULT_CONFIG.normalization.separator_chars)}]")


def value_to_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        return " ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)


def normalize_text(text: Any) -> str:
    if text is None:
        cleaned_text = ""
    elif not isinstance(text, str) and pd.isna(text):
        cleaned_text = ""
    else:
        cleaned_text = str(text)

    normalization = DEFAULT_CONFIG.normalization
    if normalization.replace_separators:
        cleaned_text = _SEPARATOR_RE.sub(" ", cleaned_text)
    if normalization.lowercase:
        cleaned_text = cleaned_text.lower()
    if normalization.collapse_whitespace:
        cleaned_text = _WHITESPACE_RE.sub(" ", cleaned_text)
    if normalization.strip:
        cleaned_text = cleaned_text.strip()
    return cleaned_text


def build_content_frame(frame: pd.DataFrame, text_columns: Sequence[str]) -> pd.DataFrame:
    require_columns(frame, ["id"], "Input frame")
    output_frame = frame.copy()
    content_series = pd.Series([""] * len(output_frame), index=output_frame.index, dtype="object")

    for column in text_columns:
        if column in output_frame.columns:
            column_text = output_frame[column].map(value_to_text)
        else:
            column_text = pd.Series([""] * len(output_frame), index=output_frame.index, dtype="object")
        content_series = content_series.str.cat(column_text, sep=" ")

    normalization = DEFAULT_CONFIG.normalization
    content_series = content_series.astype(str)
    if normalization.replace_separators:
        content_series = content_series.str.replace(_SEPARATOR_RE, " ", regex=True)
    if normalization.lowercase:
        content_series = content_series.str.lower()
    if normalization.collapse_whitespace:
        content_series = content_series.str.replace(_WHITESPACE_RE, " ", regex=True)
    if normalization.strip:
        content_series = content_series.str.strip()

    output_frame["content"] = content_series
    output_frame["id"] = output_frame["id"].astype(str)
    return output_frame


def tokenize(text: str) -> list[str]:
    return _TOKEN_RE.findall(normalize_text(text))


def build_query_classifier_frame(query_frame: pd.DataFrame, include_tags: bool | None = None) -> pd.DataFrame:
    use_tags = DEFAULT_CONFIG.data_columns.use_query_tags_in_classifier if include_tags is None else include_tags
    columns = ["title", "text"]
    if use_tags and "tags" in query_frame.columns:
        columns.append("tags")
    return build_content_frame(query_frame, columns)


# --- End src/data/text.py ---


# --- Begin src/evaluation/metrics.py ---


import json
from pathlib import Path

import numpy as np



def load_ground_truth(path: Path) -> dict[str, GroundTruthEntry]:
    if not path.exists():
        raise FileNotFoundError(f"Ground-truth file not found: {path}")
    with open(path, "r", encoding="utf-8") as handle:
        raw_ground_truth = json.load(handle)
    ground_truth: dict[str, GroundTruthEntry] = {}
    for query_id, info in raw_ground_truth.items():
        relevant_items = info.get("relevant_doc_ids", [])
        ground_truth[str(query_id)] = {
            "relevant_doc_ids": {str(item["doc_id"]) for item in relevant_items},
            "total_relevant_docs": int(info.get("total_relevant_docs", len(relevant_items))),
            "category": info.get("category"),
        }
    return ground_truth


def recall_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    recalls: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue
        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        total_relevant_docs = ground_truth[query_id]["total_relevant_docs"]
        predicted_doc_ids = item["relevant_docs"][:k]
        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        recalls.append(hits / total_relevant_docs if total_relevant_docs > 0 else 0.0)
    return float(np.mean(recalls)) if recalls else 0.0


def precision_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    precisions: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue
        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        predicted_doc_ids = item["relevant_docs"][:k]
        if not predicted_doc_ids:
            precisions.append(0.0)
            continue
        hits = sum(doc_id in relevant_doc_ids for doc_id in predicted_doc_ids)
        precisions.append(hits / len(predicted_doc_ids))
    return float(np.mean(precisions)) if precisions else 0.0


def mrr_at_k(results: list[RetrievalResult], ground_truth: dict[str, GroundTruthEntry], k: int) -> float:
    reciprocal_ranks: list[float] = []
    for item in results:
        query_id = str(item["query_id"])
        if query_id not in ground_truth:
            continue
        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"]
        reciprocal_rank = 0.0
        for rank, doc_id in enumerate(item["relevant_docs"][:k], start=1):
            if doc_id in relevant_doc_ids:
                reciprocal_rank = 1.0 / rank
                break
        reciprocal_ranks.append(reciprocal_rank)
    return float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0


def compute_category_accuracy(
    ground_truth: dict[str, GroundTruthEntry],
    predicted_categories: dict[str, str] | None,
    default_if_missing: float = 0.0,
) -> float:
    if predicted_categories is None:
        return float(default_if_missing)
    try:
        from sklearn.metrics import accuracy_score
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc
    y_true: list[str] = []
    y_pred: list[str] = []
    for query_id, info in ground_truth.items():
        true_category = info.get("category")
        predicted_category = predicted_categories.get(str(query_id))
        if true_category is None or predicted_category is None:
            continue
        y_true.append(str(true_category))
        y_pred.append(str(predicted_category))
    if not y_true:
        return float(default_if_missing)
    return float(accuracy_score(y_true, y_pred))


def leaderboard_score(
    results: list[RetrievalResult],
    ground_truth: dict[str, GroundTruthEntry],
    k: int,
    predicted_categories: dict[str, str] | None = None,
    accuracy_value: float | None = None,
) -> dict[str, float]:
    recall_value = recall_at_k(results, ground_truth, k=k)
    precision_value = precision_at_k(results, ground_truth, k=k)
    mrr_value = mrr_at_k(results, ground_truth, k=k)
    category_accuracy = (
        float(accuracy_value)
        if accuracy_value is not None
        else compute_category_accuracy(ground_truth, predicted_categories)
    )
    combined_score = 0.25 * (recall_value + precision_value + mrr_value + category_accuracy)
    return {
        "Recall": recall_value,
        "Precision": precision_value,
        "MRR": mrr_value,
        "Accuracy": category_accuracy,
        "LeaderboardScore": combined_score,
    }


# --- End src/evaluation/metrics.py ---


# --- Begin src/infra/runtime.py ---


import os
from pathlib import Path



def detect_runtime_environment() -> str:
    try:
        import google.colab  # type: ignore  # noqa: F401

        return "colab"
    except Exception:
        if Path("/kaggle/input").exists():
            return "kaggle"
        return "local"


def find_kaggle_data_dir() -> Path | None:
    for dirname, _, filenames in os.walk("/kaggle/input"):
        if "docs.json" in filenames:
            return Path(dirname)
    return None


def find_colab_project_dir(project_name: str = "retrieval_project") -> Path | None:
    drive_candidates = [
        Path("/content/drive/MyDrive"),
        Path("/content/drive/Shareddrives"),
    ]

    for drive_root in drive_candidates:
        if not drive_root.exists():
            continue
        direct_candidate = drive_root / project_name
        if (direct_candidate / "data" / "docs.json").exists():
            return direct_candidate
        for candidate in drive_root.rglob(project_name):
            if candidate.is_dir() and (candidate / "data" / "docs.json").exists():
                return candidate
    return None


def find_local_project_data_dir() -> Path | None:
    roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in roots:
        data_dir = root / "data"
        if (data_dir / "docs.json").exists():
            return data_dir
    return None


def resolve_runtime_paths(output_filename: str = "solutions_SeaFour.csv") -> RuntimePaths:
    runtime_env = detect_runtime_environment()
    project_dir: Path | None = None
    work_dir = Path.cwd()


    data_dir = find_kaggle_data_dir()
    if data_dir is None:
        raise FileNotFoundError("Kaggle environment detected, but `docs.json` was not found under /kaggle/input.")
    project_dir = Path.cwd()

    cache_dir = work_dir / "cache"
    output_path = work_dir / output_filename
    return RuntimePaths(
        runtime_env=runtime_env,
        project_dir=project_dir or work_dir,
        work_dir=work_dir,
        data_dir=data_dir,
        cache_dir=cache_dir,
        output_path=output_path,
    )


# --- End src/infra/runtime.py ---


# --- Begin src/models/embeddings.py ---


from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd



def _load_sentence_model(model_name: str, cache_dir: Path, config: AllConfig = DEFAULT_CONFIG) -> Any:
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `sentence_transformers`. Install it with `%pip install sentence-transformers`."
        ) from exc

    if model_name in MODEL_MEMORY_CACHE:
        return MODEL_MEMORY_CACHE[model_name]

    model_cache_dir = ensure_cache_dirs(cache_dir)["sentence_transformers"]
    safe_model_name = _safe_component(model_name)
    local_model_dir = model_cache_dir / safe_model_name
    if local_model_dir.exists():
        print(f"Loading model weights from cache: {local_model_dir}")
        model = SentenceTransformer(str(local_model_dir))
    else:
        print(f"Downloading model weights: {model_name}")
        model = SentenceTransformer(model_name)
        local_model_dir.mkdir(parents=True, exist_ok=True)
        model.save(str(local_model_dir))
        print(f"Saved model weights to cache: {local_model_dir}")

    MODEL_MEMORY_CACHE[model_name] = model
    return model


def _load_or_encode_embeddings(
    frame: pd.DataFrame,
    kind: str,
    model: Any,
    model_name: str,
    batch_size: int,
    cache_dir: Path,
    config: AllConfig = DEFAULT_CONFIG,
) -> np.ndarray:
    directories = ensure_cache_dirs(cache_dir)
    signature = _dataframe_fingerprint(frame, ["id", "content"])
    normalization_signature = _normalization_signature(config)
    cache_name = f"{kind}_{_safe_component(model_name)}_{normalization_signature}_{signature}.npy"
    cache_path = directories["embeddings"] / cache_name
    memory_key = str(cache_path.resolve())

    if memory_key in ARRAY_MEMORY_CACHE:
        return ARRAY_MEMORY_CACHE[memory_key]

    if config.retrieval_pipeline.enable_embedding_cache and cache_path.exists():
        print(f"Loading {kind} embeddings from cache: {cache_path.name}")
        embeddings = np.load(cache_path)
    else:
        print(f"Encoding {len(frame):,} {kind} rows...")
        embeddings = model.encode(
            frame["content"].tolist(),
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embeddings = np.asarray(embeddings, dtype=np.float32)
        if config.retrieval_pipeline.enable_embedding_cache:
            np.save(cache_path, embeddings)
            print(f"Saved {kind} embeddings to cache: {cache_path.name}")

    ARRAY_MEMORY_CACHE[memory_key] = embeddings
    return embeddings


# --- End src/models/embeddings.py ---


# --- Begin src/retrieval/indexes.py ---


from pathlib import Path
from typing import Any

import pandas as pd



def _tfidf_param_candidates(config: AllConfig = DEFAULT_CONFIG) -> list[dict[str, Any]]:
    candidates = [dict(config.tfidf.__dict__)]
    min_df = config.tfidf.min_df
    if isinstance(min_df, int) and min_df > 1:
        fallback = dict(config.tfidf.__dict__)
        fallback["min_df"] = 1
        candidates.append(fallback)
    return candidates


def build_or_load_tfidf_index(
    docs_frame: pd.DataFrame,
    cache_dir: Path,
    config: AllConfig = DEFAULT_CONFIG,
) -> dict[str, Any]:
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    directories = ensure_cache_dirs(cache_dir)
    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature(config)
    doc_ids = docs_frame["id"].to_numpy()

    for params in _tfidf_param_candidates(config):
        cache_key = _hash_payload(
            {
                "docs_signature": docs_signature,
                "normalization_signature": normalization_signature,
                "tfidf_params": params,
            }
        )
        cache_path = directories["tfidf"] / f"tfidf_{cache_key}.pkl"
        memory_key = str(cache_path.resolve())
        if memory_key in OBJECT_MEMORY_CACHE:
            return OBJECT_MEMORY_CACHE[memory_key]
        if config.retrieval_pipeline.enable_classic_cache and cache_path.exists():
            print(f"Loading TF-IDF artifacts from cache: {cache_path.name}")
            artifacts = _load_pickle(cache_path)
            OBJECT_MEMORY_CACHE[memory_key] = artifacts
            return artifacts

    params = dict(config.tfidf.__dict__)
    vectorizer = TfidfVectorizer(**params)
    try:
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])
    except ValueError as err:
        if "After pruning, no terms remain" not in str(err) or params.get("min_df", 1) == 1:
            raise
        params["min_df"] = 1
        vectorizer = TfidfVectorizer(**params)
        doc_vectors = vectorizer.fit_transform(docs_frame["content"])

    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "tfidf_params": params,
        }
    )
    cache_path = directories["tfidf"] / f"tfidf_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())
    artifacts = {
        "vectorizer": vectorizer,
        "doc_vectors": doc_vectors,
        "doc_ids": doc_ids,
        "params": params,
    }
    if config.retrieval_pipeline.enable_classic_cache:
        _save_pickle(cache_path, artifacts)
        print(f"Saved TF-IDF artifacts to cache: {cache_path.name}")
    OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def build_or_load_bm25_index(
    docs_frame: pd.DataFrame,
    cache_dir: Path,
    config: AllConfig = DEFAULT_CONFIG,
) -> dict[str, Any]:
    try:
        from rank_bm25 import BM25Plus
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `rank_bm25`. Install it with `%pip install rank-bm25`."
        ) from exc

    directories = ensure_cache_dirs(cache_dir)
    docs_signature = _dataframe_fingerprint(docs_frame, ["id", "content"])
    normalization_signature = _normalization_signature(config)
    cache_key = _hash_payload(
        {
            "docs_signature": docs_signature,
            "normalization_signature": normalization_signature,
            "bm25_params": config.bm25.__dict__,
        }
    )
    cache_path = directories["bm25"] / f"bm25_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())
    if memory_key in OBJECT_MEMORY_CACHE:
        return OBJECT_MEMORY_CACHE[memory_key]
    if config.retrieval_pipeline.enable_classic_cache and cache_path.exists():
        print(f"Loading BM25 index from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    tokenized_corpus = [tokenize(text) for text in docs_frame["content"]]
    bm25 = BM25Plus(tokenized_corpus, **config.bm25.__dict__)
    artifacts = {"bm25": bm25, "doc_ids": docs_frame["id"].to_numpy()}
    if config.retrieval_pipeline.enable_classic_cache:
        _save_pickle(cache_path, artifacts)
        print(f"Saved BM25 index to cache: {cache_path.name}")
    OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


# --- End src/retrieval/indexes.py ---


# --- Begin src/retrieval/search.py ---


import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd



def validate_pipeline_settings(document_count: int, config: AllConfig = DEFAULT_CONFIG) -> None:
    final_model = config.retrieval_pipeline.final_model
    evaluation_models = config.retrieval_pipeline.evaluation_models
    if final_model not in {"tfidf", "bm25", "embedding"}:
        raise ValueError(f"Unknown final_model: {final_model}")
    if any(model_name not in {"tfidf", "bm25", "embedding"} for model_name in evaluation_models):
        raise ValueError(f"Unknown model in evaluation_models: {evaluation_models}")
    if document_count <= 0:
        raise ValueError("The document collection is empty.")
    if config.retrieval_pipeline.submit_top_k <= 0:
        raise ValueError("submit_top_k must be positive.")


def top_k_indices(score_vector: np.ndarray, top_k: int) -> np.ndarray:
    if top_k <= 0:
        raise ValueError("top_k must be positive.")
    capped_top_k = min(top_k, score_vector.shape[0])
    if capped_top_k == score_vector.shape[0]:
        return np.argsort(score_vector)[::-1]
    candidate_indices = np.argpartition(score_vector, -capped_top_k)[-capped_top_k:]
    return candidate_indices[np.argsort(score_vector[candidate_indices])[::-1]]


def truncate_results(results: list[RetrievalResult], top_k: int) -> list[RetrievalResult]:
    if top_k <= 0:
        raise ValueError("top_k must be positive.")
    return [{"query_id": item["query_id"], "relevant_docs": item["relevant_docs"][:top_k]} for item in results]


def progress_interval(total_items: int, target_updates: int = 5) -> int:
    return max(1, total_items // max(1, target_updates))


def prepare_retriever(
    model_name: ModelName,
    docs_frame: pd.DataFrame,
    cache_dir: Path,
    config: AllConfig = DEFAULT_CONFIG,
) -> dict[str, Any]:
    if model_name == "tfidf":
        return build_or_load_tfidf_index(docs_frame, cache_dir=cache_dir, config=config)
    if model_name == "bm25":
        return build_or_load_bm25_index(docs_frame, cache_dir=cache_dir, config=config)
    if model_name == "embedding":
        model = _load_sentence_model(config.embedding.model_name, cache_dir=cache_dir, config=config)
        doc_embeddings = _load_or_encode_embeddings(
            docs_frame,
            kind="docs",
            model=model,
            model_name=config.embedding.model_name,
            batch_size=config.embedding.batch_size,
            cache_dir=cache_dir,
            config=config,
        )
        return {"model": model, "doc_embeddings": doc_embeddings, "doc_ids": docs_frame["id"].to_numpy()}
    raise ValueError(f"Unknown model: {model_name}")


def run_tfidf_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    cache_dir: Path,
    prepared_artifacts: dict[str, Any] | None = None,
    config: AllConfig = DEFAULT_CONFIG,
) -> list[RetrievalResult]:
    artifacts = prepared_artifacts or build_or_load_tfidf_index(docs_frame, cache_dir=cache_dir, config=config)
    vectorizer = artifacts["vectorizer"]
    doc_vectors = artifacts["doc_vectors"]
    doc_ids = artifacts["doc_ids"]
    query_vectors = vectorizer.transform(queries_frame["content"])
    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    log_every = progress_interval(len(query_ids))
    print(f"  [TF-IDF] vectorized {len(query_ids):,} queries against {len(doc_ids):,} docs with capped_top_k={capped_top_k:,}")
    results: list[RetrievalResult] = []
    for row_index, query_id in enumerate(query_ids):
        score_row = query_vectors[row_index] @ doc_vectors.T
        score_vector = np.asarray(score_row.toarray()).ravel()
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append({"query_id": query_id, "relevant_docs": doc_ids[top_indices].tolist()})
        if (row_index + 1) % log_every == 0 or row_index == len(query_ids) - 1:
            print(f"  [TF-IDF] processed {row_index + 1:,}/{len(query_ids):,} queries")
    return results


def run_bm25_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    cache_dir: Path,
    prepared_artifacts: dict[str, Any] | None = None,
    config: AllConfig = DEFAULT_CONFIG,
) -> list[RetrievalResult]:
    artifacts = prepared_artifacts or build_or_load_bm25_index(docs_frame, cache_dir=cache_dir, config=config)
    bm25 = artifacts["bm25"]
    doc_ids = artifacts["doc_ids"]
    capped_top_k = min(top_k, len(doc_ids))
    query_pairs = list(queries_frame[["id", "content"]].itertuples(index=False, name=None))
    log_every = progress_interval(len(query_pairs))
    print(f"  [BM25+] scoring {len(query_pairs):,} queries against {len(doc_ids):,} docs with capped_top_k={capped_top_k:,}")
    results: list[RetrievalResult] = []
    for row_index, (query_id, query_text) in enumerate(query_pairs):
        score_vector = np.asarray(bm25.get_scores(query_text.split()), dtype=np.float32)
        top_indices = top_k_indices(score_vector, capped_top_k)
        results.append({"query_id": str(query_id), "relevant_docs": doc_ids[top_indices].tolist()})
        if (row_index + 1) % log_every == 0 or row_index == len(query_pairs) - 1:
            print(f"  [BM25+] processed {row_index + 1:,}/{len(query_pairs):,} queries")
    return results


def run_embedding_search(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    cache_dir: Path,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
    config: AllConfig = DEFAULT_CONFIG,
) -> list[RetrievalResult]:
    artifacts = prepared_artifacts or prepare_retriever("embedding", docs_frame, cache_dir=cache_dir, config=config)
    model = artifacts["model"]
    doc_embeddings = artifacts["doc_embeddings"]
    doc_ids = artifacts["doc_ids"]
    query_embeddings = _load_or_encode_embeddings(
        queries_frame,
        kind=embedding_kind,
        model=model,
        model_name=config.embedding.model_name,
        batch_size=config.embedding.batch_size,
        cache_dir=cache_dir,
        config=config,
    )
    query_ids = queries_frame["id"].astype(str).tolist()
    capped_top_k = min(top_k, len(doc_ids))
    results: list[RetrievalResult] = []
    chunk_size = config.embedding.query_chunk_size
    total_chunks = (len(query_embeddings) + chunk_size - 1) // chunk_size
    print(f"  [Embedding] scoring {len(query_ids):,} queries against {len(doc_ids):,} docs with capped_top_k={capped_top_k:,}, chunk_size={chunk_size:,}, embedding_cache_key='{embedding_kind}'")
    for chunk_index, start_index in enumerate(range(0, len(query_embeddings), chunk_size), start=1):
        stop_index = start_index + chunk_size
        print(f"  [Embedding] chunk {chunk_index:,}/{total_chunks:,}: queries {start_index + 1:,}-{min(stop_index, len(query_embeddings)):,}")
        score_block = query_embeddings[start_index:stop_index] @ doc_embeddings.T
        for row_offset, score_vector in enumerate(score_block):
            top_indices = top_k_indices(score_vector, capped_top_k)
            query_id = query_ids[start_index + row_offset]
            results.append({"query_id": query_id, "relevant_docs": doc_ids[top_indices].tolist()})
    return results


def run_retrieval(
    model_name: ModelName,
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    top_k: int,
    cache_dir: Path,
    prepared_artifacts: dict[str, Any] | None = None,
    embedding_kind: str = "queries",
    config: AllConfig = DEFAULT_CONFIG,
) -> list[RetrievalResult]:
    models: dict[ModelName, Any] = {
        "tfidf": run_tfidf_search,
        "bm25": run_bm25_search,
        "embedding": run_embedding_search,
    }
    if model_name not in models:
        raise ValueError(f"Unknown model '{model_name}'. Choose from {list(models)}")
    print("=" * 88)
    print(f"Starting retrieval: model={model_name}")
    print(
        f"  parameters: top_k={top_k:,}, docs={len(docs_frame):,}, queries={len(queries_frame):,}, "
        f"prepared_artifacts={'yes' if prepared_artifacts is not None else 'no'}, embedding_kind='{embedding_kind}'"
    )
    start_time = time.time()
    if model_name == "embedding":
        results = models[model_name](
            docs_frame, queries_frame, top_k=top_k, cache_dir=cache_dir,
            prepared_artifacts=prepared_artifacts, embedding_kind=embedding_kind, config=config
        )
    else:
        results = models[model_name](
            docs_frame, queries_frame, top_k=top_k, cache_dir=cache_dir,
            prepared_artifacts=prepared_artifacts, config=config
        )
    elapsed_seconds = time.time() - start_time
    print(f"Completed retrieval: model={model_name}, results={len(results):,} queries, elapsed={elapsed_seconds:.1f}s")
    print("=" * 88)
    return results


def run_category_filtered_retrieval(
    docs_frame: pd.DataFrame,
    queries_frame: pd.DataFrame,
    classifier_artifacts: dict[str, Any],
    top_k: int,
    cache_dir: Path,
    model_name: str | None = None,
    embedding_kind_prefix: str = "queries",
    config: AllConfig = DEFAULT_CONFIG,
) -> tuple[list[RetrievalResult], dict[str, str]]:

    embedding_model_name = model_name or config.embedding.model_name
    query_category_map = predict_category_map(queries_frame, classifier_artifacts)

    category_to_positions: dict[str, list[int]] = {}
    for iloc_idx, (_, row) in enumerate(docs_frame.iterrows()):
        category_to_positions.setdefault(row["category"], []).append(iloc_idx)

    model = _load_sentence_model(embedding_model_name, cache_dir=cache_dir, config=config)
    full_doc_embeddings = _load_or_encode_embeddings(
        docs_frame,
        "docs",
        model,
        embedding_model_name,
        config.embedding.batch_size,
        cache_dir,
        config,
    )

    results: list[RetrievalResult] = []
    for category in set(query_category_map.values()):
        cat_positions_list = category_to_positions.get(category, [])
        cat_queries_frame = queries_frame[queries_frame["id"].map(query_category_map) == category]
        if not cat_positions_list:
            print(f"WARNING: No documents found for predicted category '{category}', falling back to full corpus for those queries.")
            cat_doc_embeddings = full_doc_embeddings
            cat_docs_frame = docs_frame
        else:
            cat_positions = np.array(cat_positions_list)
            cat_doc_embeddings = full_doc_embeddings[cat_positions]
            cat_docs_frame = docs_frame.iloc[cat_positions].reset_index(drop=True)

        embedding_kind = f"{embedding_kind_prefix}_{category}_filtered"
        cat_query_embeddings = _load_or_encode_embeddings(
            cat_queries_frame,
            embedding_kind,
            model,
            embedding_model_name,
            config.embedding.batch_size,
            cache_dir,
            config,
        )
        scores = cat_query_embeddings @ cat_doc_embeddings.T
        actual_top_k = min(top_k, len(cat_docs_frame))
        for q_idx, query_row in enumerate(cat_queries_frame.itertuples()):
            query_scores = scores[q_idx]
            local_top_k = min(actual_top_k, len(query_scores))
            top_local_indices = top_k_indices(query_scores, local_top_k)
            relevant_docs = [str(cat_docs_frame.iloc[li]["id"]) for li in top_local_indices]
            results.append({"query_id": str(query_row.id), "relevant_docs": relevant_docs})

    query_order = list(queries_frame["id"].astype(str))
    results_by_query = {item["query_id"]: item for item in results}
    ordered_results = [results_by_query[query_id] for query_id in query_order if query_id in results_by_query]
    return ordered_results, query_category_map


# --- End src/retrieval/search.py ---


# --- Begin src/categorization/classifier.py ---


from pathlib import Path
from typing import Any

import pandas as pd



def build_or_load_category_classifier(
    train_frame: pd.DataFrame,
    cache_dir: Path,
    config: AllConfig = DEFAULT_CONFIG,
) -> dict[str, Any]:
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.svm import LinearSVC
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `scikit-learn`. Install it with `%pip install scikit-learn`."
        ) from exc

    directories = ensure_cache_dirs(cache_dir)
    require_columns(train_frame, ["id", "content", "category"], "Classifier training data")
    train_signature = _dataframe_fingerprint(train_frame, ["id", "content", "category"])
    normalization_signature = _normalization_signature(config)
    tfidf_params = dict(config.tfidf.__dict__)
    cache_key = _hash_payload(
        {
            "train_signature": train_signature,
            "normalization_signature": normalization_signature,
            "classifier_tfidf_params": tfidf_params,
        }
    )
    cache_path = directories["classifier"] / f"category_classifier_{cache_key}.pkl"
    memory_key = str(cache_path.resolve())
    if memory_key in OBJECT_MEMORY_CACHE:
        return OBJECT_MEMORY_CACHE[memory_key]
    if config.retrieval_pipeline.enable_classic_cache and cache_path.exists():
        print(f"Loading category classifier from cache: {cache_path.name}")
        artifacts = _load_pickle(cache_path)
        OBJECT_MEMORY_CACHE[memory_key] = artifacts
        return artifacts

    vectorizer = TfidfVectorizer(**tfidf_params)
    train_vectors = vectorizer.fit_transform(train_frame["content"])
    classifier = LinearSVC()
    classifier.fit(train_vectors, train_frame["category"].astype(str))
    artifacts = {"vectorizer": vectorizer, "classifier": classifier}
    if config.retrieval_pipeline.enable_classic_cache:
        _save_pickle(cache_path, artifacts)
        print(f"Saved category classifier to cache: {cache_path.name}")
    OBJECT_MEMORY_CACHE[memory_key] = artifacts
    return artifacts


def predict_category_map(query_frame: pd.DataFrame, classifier_artifacts: dict[str, Any]) -> dict[str, str]:
    query_vectors = classifier_artifacts["vectorizer"].transform(query_frame["content"])
    predictions = classifier_artifacts["classifier"].predict(query_vectors)
    return {
        str(query_id): str(prediction)
        for query_id, prediction in zip(query_frame["id"].astype(str), predictions)
    }


def build_doc_category_map(docs_frame: pd.DataFrame) -> dict[str, Any]:
    require_columns(docs_frame, ["id", "category"], "Documents frame")
    return (
        docs_frame[["id", "category"]]
        .assign(id=lambda frame: frame["id"].astype(str))
        .set_index("id")["category"]
        .to_dict()
    )


# --- End src/categorization/classifier.py ---


# --- Begin src/cross_encoder/training.py ---


from pathlib import Path
from typing import Any, Sequence

import numpy as np
import pandas as pd



def build_text_map(frame: pd.DataFrame, id_column: str = "id", text_column: str = "content") -> dict[str, str]:
    require_columns(frame, [id_column, text_column], f"Frame[{id_column}, {text_column}]")
    return {
        str(row_id): str(text)
        for row_id, text in frame[[id_column, text_column]].itertuples(index=False, name=None)
    }


def sample_random_negative_doc_ids(
    all_doc_ids: Sequence[str],
    excluded_doc_ids: set[str],
    sample_size: int,
    rng: Any,
) -> list[str]:
    if sample_size <= 0 or not all_doc_ids:
        return []
    negatives: list[str] = []
    max_attempts = max(100, sample_size * 50)
    attempts = 0
    while len(negatives) < sample_size and attempts < max_attempts:
        candidate = str(all_doc_ids[rng.randrange(len(all_doc_ids))])
        attempts += 1
        if candidate in excluded_doc_ids or candidate in negatives:
            continue
        negatives.append(candidate)
    return negatives


def mine_hard_negative_doc_ids(
    train_queries_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, GroundTruthEntry],
    query_ids: Sequence[str],
    cache_dir: Path,
    top_k: int | None = None,
    prepared_artifacts: dict[str, Any] | None = None,
    config: AllConfig = DEFAULT_CONFIG,
) -> dict[str, list[str]]:
    hard_negative_top_k = top_k or config.cross_encoder.hard_negative_top_k
    if hard_negative_top_k <= 0:
        return {}
    query_id_set = {str(query_id) for query_id in query_ids}
    mining_queries_frame = (
        train_queries_frame.assign(id=train_queries_frame["id"].astype(str))
        .loc[lambda frame: frame["id"].isin(query_id_set), ["id", "content"]]
        .reset_index(drop=True)
    )
    if mining_queries_frame.empty:
        return {}
    embedding_artifacts = prepared_artifacts or prepare_retriever("embedding", docs_frame, cache_dir=cache_dir, config=config)
    retrieval_results = run_retrieval(
        model_name="embedding",
        docs_frame=docs_frame,
        queries_frame=mining_queries_frame,
        top_k=hard_negative_top_k,
        cache_dir=cache_dir,
        prepared_artifacts=embedding_artifacts,
        embedding_kind="queries_train_hardneg",
        config=config,
    )
    hard_negative_doc_ids_by_query: dict[str, list[str]] = {}
    for item in retrieval_results:
        query_id = str(item["query_id"])
        relevant_doc_ids = ground_truth[query_id]["relevant_doc_ids"] if query_id in ground_truth else set()
        negatives: list[str] = []
        seen_doc_ids: set[str] = set()
        for doc_id in item["relevant_docs"]:
            candidate_doc_id = str(doc_id)
            if candidate_doc_id in relevant_doc_ids or candidate_doc_id in seen_doc_ids:
                continue
            negatives.append(candidate_doc_id)
            seen_doc_ids.add(candidate_doc_id)
        hard_negative_doc_ids_by_query[query_id] = negatives
    counts = [len(doc_ids) for doc_ids in hard_negative_doc_ids_by_query.values()]
    if counts:
        print(
            f"  [HardNegatives] mined for {len(counts):,} queries "
            f"(per-query min/mean/max={min(counts):,}/{float(np.mean(counts)):.1f}/{max(counts):,}, top_k={hard_negative_top_k:,})"
        )
    return hard_negative_doc_ids_by_query


def build_cross_encoder_training_examples(
    query_text_map: dict[str, str],
    doc_text_map: dict[str, str],
    ground_truth: dict[str, GroundTruthEntry],
    query_ids: Sequence[str],
    max_positives_per_query: int | None = None,
    negatives_per_positive: int | None = None,
    hard_negative_doc_ids_by_query: dict[str, list[str]] | None = None,
    seed: int | None = None,
    config: AllConfig = DEFAULT_CONFIG,
) -> list[Any]:
    try:
        from sentence_transformers import InputExample
    except ImportError as exc:
        raise ImportError(
            "Missing dependency `sentence_transformers`. Install it with `%pip install sentence-transformers`."
        ) from exc

    import random

    max_positives = max_positives_per_query or config.cross_encoder.max_positives_per_query
    negatives = negatives_per_positive or config.cross_encoder.negatives_per_positive
    rng = random.Random(config.cross_encoder.random_seed if seed is None else seed)
    all_doc_ids = list(doc_text_map.keys())
    examples: list[Any] = []
    hard_negative_count = 0
    random_negative_count = 0
    for query_id in query_ids:
        query_id_str = str(query_id)
        query_text = query_text_map.get(query_id_str)
        if query_text is None or query_id_str not in ground_truth:
            continue
        relevant_doc_ids = [
            str(doc_id)
            for doc_id in ground_truth[query_id_str]["relevant_doc_ids"]
            if str(doc_id) in doc_text_map
        ]
        if not relevant_doc_ids:
            continue
        rng.shuffle(relevant_doc_ids)
        selected_positive_doc_ids = relevant_doc_ids[:max_positives]
        relevant_doc_id_set = set(relevant_doc_ids)
        hard_negative_pool = [
            doc_id
            for doc_id in (hard_negative_doc_ids_by_query or {}).get(query_id_str, [])
            if doc_id in doc_text_map and doc_id not in relevant_doc_id_set
        ]
        for positive_doc_id in selected_positive_doc_ids:
            examples.append(InputExample(texts=[query_text, doc_text_map[positive_doc_id]], label=1.0))
            negative_doc_ids: list[str] = []
            if hard_negative_pool:
                hard_take = min(negatives, len(hard_negative_pool))
                negative_doc_ids.extend(rng.sample(hard_negative_pool, hard_take))
                hard_negative_count += hard_take
            if len(negative_doc_ids) < negatives:
                random_needed = negatives - len(negative_doc_ids)
                extra_random_negatives = sample_random_negative_doc_ids(
                    all_doc_ids=all_doc_ids,
                    excluded_doc_ids=relevant_doc_id_set | set(negative_doc_ids),
                    sample_size=random_needed,
                    rng=rng,
                )
                negative_doc_ids.extend(extra_random_negatives)
                random_negative_count += len(extra_random_negatives)
            for negative_doc_id in negative_doc_ids:
                examples.append(InputExample(texts=[query_text, doc_text_map[negative_doc_id]], label=0.0))
    print(
        f"Cross-encoder pairs: total={len(examples):,}, "
        f"hard_negatives={hard_negative_count:,}, random_negatives={random_negative_count:,}"
    )
    return examples


# --- End src/cross_encoder/training.py ---


# --- Begin src/cross_encoder/inference.py ---


from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd



def build_or_load_cross_encoder(
    train_queries_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, GroundTruthEntry],
    cache_dir: Path,
    config: AllConfig = DEFAULT_CONFIG,
) -> Any:
    try:
        from sentence_transformers import CrossEncoder
        from torch.utils.data import DataLoader
        import torch
    except ImportError as exc:
        raise ImportError(
            "Missing dependencies for cross-encoder training. Install `%pip install sentence-transformers torch`."
        ) from exc

    directories = ensure_cache_dirs(cache_dir)
    query_text_map = build_text_map(train_queries_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")
    candidate_query_ids = [
        query_id
        for query_id in train_queries_frame["id"].astype(str).tolist()
        if query_id in ground_truth and query_id in query_text_map
    ]
    if config.cross_encoder.train_query_limit > 0:
        candidate_query_ids = candidate_query_ids[: config.cross_encoder.train_query_limit]
    if not candidate_query_ids:
        raise ValueError("No training queries available for cross-encoder training.")
    train_signature = _hash_payload(
        {
            "query_signature": _dataframe_fingerprint(train_queries_frame, ["id", "content"]),
            "doc_signature": _dataframe_fingerprint(docs_frame, ["id", "content"]),
            "query_count": len(candidate_query_ids),
            "model": config.cross_encoder.model_name,
            "epochs": config.cross_encoder.epochs,
            "batch_size": config.cross_encoder.batch_size,
            "max_length": config.cross_encoder.max_length,
            "max_pos_per_query": config.cross_encoder.max_positives_per_query,
            "neg_per_pos": config.cross_encoder.negatives_per_positive,
            "hard_neg_top_k": config.cross_encoder.hard_negative_top_k,
            "hard_neg_model": "embedding",
            "hard_neg_embedding_model": config.embedding.model_name,
            "seed": config.cross_encoder.random_seed,
        }
    )
    model_dir = directories["cross_encoder"] / f"{_safe_component(config.cross_encoder.model_name)}_{train_signature}"
    cache_marker = model_dir / "config.json"
    memory_key = f"cross_encoder::{model_dir.resolve()}"
    if memory_key in OBJECT_MEMORY_CACHE:
        return OBJECT_MEMORY_CACHE[memory_key]
    if config.cross_encoder.enable_cache and cache_marker.exists():
        print(f"Loading cross-encoder from cache: {model_dir.name}")
        try:
            cached_model = CrossEncoder(str(model_dir), max_length=config.cross_encoder.max_length)
            if config.cross_encoder.fp16 and torch.cuda.is_available():
                cached_model.model.half()
            OBJECT_MEMORY_CACHE[memory_key] = cached_model
            return cached_model
        except Exception as exc:
            print(f"Cross-encoder cache load failed, retraining: {exc}")
    elif config.cross_encoder.enable_cache and model_dir.exists():
        print(f"Cross-encoder cache directory exists but is incomplete: {model_dir}")

    embedding_artifacts = prepare_retriever("embedding", docs_frame, cache_dir=cache_dir, config=config)
    hard_negative_doc_ids_by_query = mine_hard_negative_doc_ids(
        train_queries_frame=train_queries_frame,
        docs_frame=docs_frame,
        ground_truth=ground_truth,
        query_ids=candidate_query_ids,
        top_k=config.cross_encoder.hard_negative_top_k,
        prepared_artifacts=embedding_artifacts,
        cache_dir=cache_dir,
        config=config,
    )
    training_examples = build_cross_encoder_training_examples(
        query_text_map=query_text_map,
        doc_text_map=doc_text_map,
        ground_truth=ground_truth,
        query_ids=candidate_query_ids,
        max_positives_per_query=config.cross_encoder.max_positives_per_query,
        negatives_per_positive=config.cross_encoder.negatives_per_positive,
        hard_negative_doc_ids_by_query=hard_negative_doc_ids_by_query,
        seed=config.cross_encoder.random_seed,
        config=config,
    )
    if not training_examples:
        raise ValueError("Cross-encoder training set is empty after preprocessing.")
    print(f"Training cross-encoder on {len(training_examples):,} pairs from {len(candidate_query_ids):,} queries")
    cross_encoder = CrossEncoder(config.cross_encoder.model_name, max_length=config.cross_encoder.max_length)
    train_loader = DataLoader(training_examples, shuffle=True, batch_size=config.cross_encoder.batch_size)
    warmup_steps = max(1, int(len(train_loader) * config.cross_encoder.epochs * 0.1))
    if config.cross_encoder.enable_cache:
        model_dir.mkdir(parents=True, exist_ok=True)
    output_path = str(model_dir) if config.cross_encoder.enable_cache else None
    cross_encoder.fit(
        train_dataloader=train_loader,
        epochs=config.cross_encoder.epochs,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
        output_path=output_path,
    )
    if config.cross_encoder.enable_cache:
        print(f"Saved cross-encoder to cache: {model_dir.name}")
        cross_encoder.save(str(model_dir))
        cached_model = CrossEncoder(str(model_dir), max_length=config.cross_encoder.max_length)
        if config.cross_encoder.fp16 and torch.cuda.is_available():
            cached_model.model.half()
        OBJECT_MEMORY_CACHE[memory_key] = cached_model
        return cached_model
    if config.cross_encoder.fp16 and torch.cuda.is_available():
        cross_encoder.model.half()
    OBJECT_MEMORY_CACHE[memory_key] = cross_encoder
    return cross_encoder


def evaluate_isolated_cross_encoder(
    cross_encoder: Any,
    query_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    ground_truth: dict[str, dict],
    hard_negative_doc_ids_by_query: dict[str, list[str]],
    sample_limit: int = 100,
) -> None:
    from sklearn.metrics import average_precision_score, roc_auc_score

    print("Running isolated Cross-Encoder diagnostics...")
    query_text_map = build_text_map(query_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")
    model_inputs = []
    labels = []
    queries_tested = 0
    for query_id, info in ground_truth.items():
        if queries_tested >= sample_limit:
            break
        query_id_str = str(query_id)
        if query_id_str not in query_text_map:
            continue
        query_text = query_text_map[query_id_str]
        relevant_doc_ids = [str(doc_id) for doc_id in info.get("relevant_doc_ids", []) if str(doc_id) in doc_text_map]
        if not relevant_doc_ids:
            continue
        hard_negatives = hard_negative_doc_ids_by_query.get(query_id_str, [])
        hard_negatives = [doc_id for doc_id in hard_negatives if doc_id in doc_text_map and doc_id not in relevant_doc_ids][:10]
        if not hard_negatives:
            continue
        for doc_id in relevant_doc_ids:
            model_inputs.append([query_text, doc_text_map[doc_id]])
            labels.append(1.0)
        for doc_id in hard_negatives:
            model_inputs.append([query_text, doc_text_map[doc_id]])
            labels.append(0.0)
        queries_tested += 1
    if not labels:
        print("Not enough valid pairs to run diagnostics.")
        return
    print(f"Scoring {len(labels)} pairs ({sum(labels)} Positives, {len(labels) - sum(labels)} Negatives)...")
    scores = np.asarray(cross_encoder.predict(model_inputs, batch_size=32, show_progress_bar=True), dtype=np.float32).reshape(-1)
    roc_auc = roc_auc_score(labels, scores)
    pr_auc = average_precision_score(labels, scores)
    print("\n=== Isolated Cross-Encoder Metrics ===")
    print(f"ROC-AUC:           {roc_auc:.4f} (1.0 is perfect separation, 0.5 is random guessing)")
    print(f"Average Precision: {pr_auc:.4f} (Higher is better, measures precision-recall curve)")
    pos_scores = scores[np.array(labels) == 1.0]
    neg_scores = scores[np.array(labels) == 0.0]
    print(f"\nMean Score (Positives): {np.mean(pos_scores):.4f}")
    print(f"Mean Score (Negatives): {np.mean(neg_scores):.4f}")


# --- End src/cross_encoder/inference.py ---


# --- Begin src/reranking/reranker.py ---


from typing import Any

import numpy as np
import pandas as pd



def compute_category_boost(
    query_id: str,
    doc_ids: list[str],
    query_category_map: dict[str, str],
    doc_category_map: dict[str, Any],
    category_bonus: float,
) -> np.ndarray:
    boosts = np.zeros(len(doc_ids), dtype=np.float32)
    predicted_query_cat = query_category_map.get(query_id)
    if predicted_query_cat is None or category_bonus == 0.0:
        return boosts
    target_category = str(predicted_query_cat)
    for index, doc_id in enumerate(doc_ids):
        raw_doc_cat = doc_category_map.get(doc_id)
        doc_category = "unknown" if raw_doc_cat is None or pd.isna(raw_doc_cat) else str(raw_doc_cat)
        if doc_category == target_category:
            boosts[index] = category_bonus
    return boosts


def rerank_results_with_cross_encoder(
    results: list[RetrievalResult],
    query_frame: pd.DataFrame,
    docs_frame: pd.DataFrame,
    cross_encoder: Any,
    query_category_map: dict[str, str],
    doc_category_map: dict[str, Any],
    infer_batch_size: int,
    rerank_top_m: int = 100,
    category_bonus: float = 2.0,
    return_diagnostics: bool = False,
) -> list[RetrievalResult] | tuple[list[RetrievalResult], list[dict[str, Any]]]:
    if rerank_top_m <= 0:
        raise ValueError("rerank_top_m must be positive.")
    query_text_map = build_text_map(query_frame, id_column="id", text_column="content")
    doc_text_map = build_text_map(docs_frame, id_column="id", text_column="content")
    all_pairs: list[list[str]] = []
    query_meta: list[dict[str, Any]] = []
    for item in results:
        query_id = str(item["query_id"])
        doc_ids = [str(doc_id) for doc_id in item["relevant_docs"]]
        if query_id not in query_text_map:
            query_meta.append(
                {"query_id": query_id, "skip": True, "skip_reason": "missing_query_text", "doc_ids": doc_ids, "head_doc_ids": doc_ids[:rerank_top_m]}
            )
            continue
        head_doc_ids = doc_ids[:rerank_top_m]
        tail_doc_ids = doc_ids[rerank_top_m:]
        scored_doc_ids = [doc_id for doc_id in head_doc_ids if doc_id in doc_text_map]
        missing_head_doc_ids = [doc_id for doc_id in head_doc_ids if doc_id not in doc_text_map]
        if len(scored_doc_ids) <= 1:
            query_meta.append(
                {"query_id": query_id, "skip": True, "skip_reason": "insufficient_scored_docs", "doc_ids": doc_ids, "head_doc_ids": head_doc_ids}
            )
            continue
        pairs = [[query_text_map[query_id], doc_text_map[doc_id]] for doc_id in scored_doc_ids]
        start = len(all_pairs)
        all_pairs.extend(pairs)
        query_meta.append(
            {
                "query_id": query_id,
                "skip": False,
                "head_doc_ids": head_doc_ids,
                "scored_doc_ids": scored_doc_ids,
                "missing_head_doc_ids": missing_head_doc_ids,
                "tail_doc_ids": tail_doc_ids,
                "slice": (start, start + len(pairs)),
            }
        )
    all_scores = (
        np.asarray(cross_encoder.predict(all_pairs, batch_size=infer_batch_size, show_progress_bar=False), dtype=np.float32)
        if all_pairs
        else np.array([], dtype=np.float32)
    )
    reranked_results: list[RetrievalResult] = []
    rerank_diagnostics: list[dict[str, Any]] = []
    reranked_query_count = 0
    for meta in query_meta:
        if meta["skip"]:
            reranked_results.append({"query_id": meta["query_id"], "relevant_docs": meta["doc_ids"]})
            if return_diagnostics:
                rerank_diagnostics.append(
                    {
                        "query_id": meta["query_id"],
                        "reranked": False,
                        "skip_reason": meta["skip_reason"],
                        "rerank_top_m": int(rerank_top_m),
                        "category_bonus": float(category_bonus),
                        "original_head_doc_ids": meta.get("head_doc_ids", []),
                        "reranked_head_doc_ids": meta.get("head_doc_ids", []),
                        "missing_head_doc_ids": [],
                        "tail_doc_ids_count": max(0, len(meta["doc_ids"]) - len(meta.get("head_doc_ids", []))),
                        "candidate_docs": [],
                    }
                )
            continue
        start, end = meta["slice"]
        ce_scores = all_scores[start:end]
        scored_doc_ids = meta["scored_doc_ids"]
        boosts = compute_category_boost(meta["query_id"], scored_doc_ids, query_category_map, doc_category_map, category_bonus)
        final_scores = ce_scores + boosts
        ranked_indices = np.argsort(final_scores)[::-1]
        reranked_scored_doc_ids = [scored_doc_ids[index] for index in ranked_indices]
        reranked_doc_ids = reranked_scored_doc_ids + meta["missing_head_doc_ids"] + meta["tail_doc_ids"]
        reranked_results.append({"query_id": meta["query_id"], "relevant_docs": reranked_doc_ids})
        reranked_query_count += 1
        if return_diagnostics:
            reranked_positions = {doc_id: rank for rank, doc_id in enumerate(reranked_scored_doc_ids, start=1)}
            candidate_docs: list[dict[str, Any]] = []
            for original_rank, doc_id in enumerate(scored_doc_ids, start=1):
                raw_doc_category = doc_category_map.get(doc_id)
                doc_category = "unknown" if raw_doc_category is None or pd.isna(raw_doc_category) else str(raw_doc_category)
                score_index = original_rank - 1
                candidate_docs.append(
                    {
                        "doc_id": doc_id,
                        "original_rank": int(original_rank),
                        "reranked_rank": int(reranked_positions[doc_id]),
                        "cross_encoder_score": float(ce_scores[score_index]),
                        "category_boost": float(boosts[score_index]),
                        "final_score": float(final_scores[score_index]),
                        "doc_category": doc_category,
                    }
                )
            rerank_diagnostics.append(
                {
                    "query_id": meta["query_id"],
                    "reranked": True,
                    "skip_reason": None,
                    "rerank_top_m": int(rerank_top_m),
                    "category_bonus": float(category_bonus),
                    "predicted_query_category": query_category_map.get(meta["query_id"]),
                    "original_head_doc_ids": meta["head_doc_ids"],
                    "reranked_head_doc_ids": reranked_scored_doc_ids + meta["missing_head_doc_ids"],
                    "missing_head_doc_ids": meta["missing_head_doc_ids"],
                    "tail_doc_ids_count": len(meta["tail_doc_ids"]),
                    "candidate_docs": candidate_docs,
                }
            )
    print(
        f"  [CrossEncoder] reranked {reranked_query_count:,}/{len(results):,} queries "
        f"(top_m={rerank_top_m:,}, category_bonus={category_bonus:.2f}, total_pairs={len(all_pairs):,})"
    )
    if return_diagnostics:
        return reranked_results, rerank_diagnostics
    return reranked_results


# --- End src/reranking/reranker.py ---


# --- Begin src/output/submission.py ---


import csv
import json
from pathlib import Path



def write_kaggle_submission(
    results: list[RetrievalResult],
    sample_csv_path: Path,
    output_csv_path: Path,
    category_predictions: dict[str, str] | None = None,
) -> None:
    prediction_map = {
        str(item["query_id"]): [str(doc_id) for doc_id in item["relevant_docs"]]
        for item in results
    }
    with open(sample_csv_path, "r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        rows = list(reader)
    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError("Invalid sample submission format.")
    query_id_column = fieldnames[0]
    prediction_column = fieldnames[1]
    category_column = fieldnames[2] if len(fieldnames) >= 3 else None
    with open(output_csv_path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            query_id = str(row[query_id_column])
            if query_id not in prediction_map:
                raise ValueError(f"Missing retrieval prediction for query_id={query_id}")
            output_row = {
                query_id_column: query_id,
                prediction_column: json.dumps(prediction_map[query_id]),
            }
            if category_column is not None:
                if category_predictions is None:
                    output_row[category_column] = row.get(category_column, "?") or "?"
                else:
                    if query_id not in category_predictions:
                        raise ValueError(f"Missing category prediction for query_id={query_id}")
                    output_row[category_column] = str(category_predictions[query_id])
            writer.writerow(output_row)


# --- End src/output/submission.py ---


# --- Begin src/pipeline.py ---


from dataclasses import dataclass
from typing import Any, Literal

import pandas as pd



@dataclass
class LoadedFrames:
    docs_raw: pd.DataFrame
    train_queries_raw: pd.DataFrame
    test_queries_raw: pd.DataFrame
    sample_submission: pd.DataFrame
    docs: pd.DataFrame
    train_queries: pd.DataFrame
    test_queries: pd.DataFrame
    docs_classifier: pd.DataFrame
    train_queries_classifier: pd.DataFrame
    test_queries_classifier: pd.DataFrame


@dataclass
class CategoryArtifacts:
    classifier_artifacts: dict[str, Any] | None
    train_query_category_map: dict[str, str] | None
    test_query_category_map: dict[str, str] | None
    doc_category_map: dict[str, Any] | None
    classifier_accuracy: float


def bootstrap(output_filename: str = "solutions_SeaFour.csv", config: AllConfig = DEFAULT_CONFIG) -> tuple[RuntimePaths, AllConfig]:
    paths = resolve_runtime_paths(output_filename=output_filename)
    paths.cache_dir.mkdir(parents=True, exist_ok=True)
    return paths, config


def load_project_frames(paths: RuntimePaths, config: AllConfig = DEFAULT_CONFIG) -> LoadedFrames:
    docs_raw_df = load_json_frame(paths.data_dir / "docs.json", "Documents")
    train_queries_raw_df = load_json_frame(paths.data_dir / "queries_train.json", "Train queries")
    test_queries_raw_df = load_json_frame(paths.data_dir / "queries_test.json", "Test queries")
    sample_submission_df = pd.read_csv(paths.data_dir / "submission.csv")

    require_columns(docs_raw_df, ["id", "title", "text", "tags", "category"], "Documents")
    require_columns(train_queries_raw_df, ["id", "title", "text", "tags", "category"], "Train queries")
    require_columns(test_queries_raw_df, ["id", "title", "text", "tags"], "Test queries")
    require_columns(sample_submission_df, ["query_id", "relevant_doc_ids", "category"], "Sample submission")
    ensure_unique_ids(docs_raw_df, "Documents")
    ensure_unique_ids(train_queries_raw_df, "Train queries")
    ensure_unique_ids(test_queries_raw_df, "Test queries")

    docs_df = build_content_frame(docs_raw_df, config.data_columns.document_text_columns)
    train_queries_df = build_content_frame(train_queries_raw_df, config.data_columns.retrieval_query_columns)
    test_queries_df = build_content_frame(test_queries_raw_df, config.data_columns.retrieval_query_columns)
    docs_classifier_df = docs_df[["id", "content", "category"]].copy()
    train_queries_classifier_df = build_query_classifier_frame(
        train_queries_raw_df,
        include_tags=config.data_columns.use_query_tags_in_classifier,
    )
    test_queries_classifier_df = build_query_classifier_frame(
        test_queries_raw_df,
        include_tags=config.data_columns.use_query_tags_in_classifier,
    )

    if docs_df["content"].eq("").all():
        raise ValueError("All document content is empty after preprocessing. Check the source columns or normalization.")

    return LoadedFrames(
        docs_raw=docs_raw_df,
        train_queries_raw=train_queries_raw_df,
        test_queries_raw=test_queries_raw_df,
        sample_submission=sample_submission_df,
        docs=docs_df,
        train_queries=train_queries_df,
        test_queries=test_queries_df,
        docs_classifier=docs_classifier_df,
        train_queries_classifier=train_queries_classifier_df,
        test_queries_classifier=test_queries_classifier_df,
    )


def prepare_retrievers(
    frames: LoadedFrames,
    paths: RuntimePaths,
    config: AllConfig = DEFAULT_CONFIG,
) -> dict[str, dict[str, Any]]:
    model_names = set(config.retrieval_pipeline.evaluation_models) | {config.retrieval_pipeline.final_model}
    prepared_retrievers: dict[str, dict[str, Any]] = {}
    for model_name in model_names:
        prepared_retrievers[model_name] = prepare_retriever(
            model_name=model_name,
            docs_frame=frames.docs,
            cache_dir=paths.cache_dir,
            config=config,
        )
    return prepared_retrievers


def predict_categories(
    frames: LoadedFrames,
    paths: RuntimePaths,
    ground_truth: dict[str, GroundTruthEntry] | None = None,
    config: AllConfig = DEFAULT_CONFIG,
) -> CategoryArtifacts:
    if not config.retrieval_pipeline.enable_category_prediction:
        return CategoryArtifacts(
            classifier_artifacts=None,
            train_query_category_map=None,
            test_query_category_map=None,
            doc_category_map=None,
            classifier_accuracy=0.0,
        )

    classifier_artifacts = build_or_load_category_classifier(
        train_frame=frames.docs_classifier,
        cache_dir=paths.cache_dir,
        config=config,
    )
    train_query_category_map = predict_category_map(frames.train_queries_classifier, classifier_artifacts)
    test_query_category_map = predict_category_map(frames.test_queries_classifier, classifier_artifacts)
    doc_category_map = build_doc_category_map(frames.docs)
    classifier_accuracy = (
        compute_category_accuracy(ground_truth, train_query_category_map)
        if ground_truth is not None
        else 0.0
    )
    return CategoryArtifacts(
        classifier_artifacts=classifier_artifacts,
        train_query_category_map=train_query_category_map,
        test_query_category_map=test_query_category_map,
        doc_category_map=doc_category_map,
        classifier_accuracy=classifier_accuracy,
    )


def build_cross_encoder_reranker(
    frames: LoadedFrames,
    paths: RuntimePaths,
    ground_truth: dict[str, GroundTruthEntry],
    config: AllConfig = DEFAULT_CONFIG,
) -> Any | None:
    if not config.retrieval_pipeline.enable_cross_encoder_rerank:
        return None
    return build_or_load_cross_encoder(
        train_queries_frame=frames.train_queries,
        docs_frame=frames.docs,
        ground_truth=ground_truth,
        cache_dir=paths.cache_dir,
        config=config,
    )


def run_first_stage_retrieval(
    frames: LoadedFrames,
    paths: RuntimePaths,
    prepared_retrievers: dict[str, dict[str, Any]],
    category_artifacts: CategoryArtifacts,
    split: Literal["train", "test"] = "test",
    top_k: int | None = None,
    config: AllConfig = DEFAULT_CONFIG,
) -> tuple[list[RetrievalResult], dict[str, str] | None]:
    query_frame = frames.train_queries if split == "train" else frames.test_queries
    embedding_kind = "queries_train" if split == "train" else "queries_test"
    retrieval_top_k = config.retrieval_pipeline.submit_top_k if top_k is None else top_k
    category_predictions = (
        category_artifacts.train_query_category_map
        if split == "train"
        else category_artifacts.test_query_category_map
    )

    if config.retrieval_pipeline.enable_category_filter and category_artifacts.classifier_artifacts is not None:
        results, _ = run_category_filtered_retrieval(
            docs_frame=frames.docs,
            queries_frame=query_frame,
            classifier_artifacts=category_artifacts.classifier_artifacts,
            top_k=retrieval_top_k,
            cache_dir=paths.cache_dir,
            config=config,
        )
        return results, category_predictions

    results = run_retrieval(
        model_name=config.retrieval_pipeline.final_model,
        docs_frame=frames.docs,
        queries_frame=query_frame,
        top_k=retrieval_top_k,
        cache_dir=paths.cache_dir,
        prepared_artifacts=prepared_retrievers[config.retrieval_pipeline.final_model],
        embedding_kind=embedding_kind,
        config=config,
    )
    return results, category_predictions


def rerank_retrieval_results(
    results: list[RetrievalResult],
    frames: LoadedFrames,
    category_artifacts: CategoryArtifacts,
    cross_encoder_reranker: Any | None,
    split: Literal["train", "test"] = "test",
    config: AllConfig = DEFAULT_CONFIG,
) -> list[RetrievalResult]:
    if cross_encoder_reranker is None:
        return results
    query_category_map = (
        category_artifacts.train_query_category_map
        if split == "train"
        else category_artifacts.test_query_category_map
    )
    if query_category_map is None or category_artifacts.doc_category_map is None:
        return results
    query_frame = frames.train_queries if split == "train" else frames.test_queries
    return rerank_results_with_cross_encoder(
        results=results,
        query_frame=query_frame,
        docs_frame=frames.docs,
        cross_encoder=cross_encoder_reranker,
        query_category_map=query_category_map,
        doc_category_map=category_artifacts.doc_category_map,
        infer_batch_size=config.cross_encoder.infer_batch_size,
        rerank_top_m=config.cross_encoder.rerank_top_m,
        category_bonus=config.cross_encoder.category_bonus if config.retrieval_pipeline.enable_category_filter else 0.0,
    )


def write_submission(
    results: list[RetrievalResult],
    paths: RuntimePaths,
    category_predictions: dict[str, str] | None = None,
) -> None:
    write_kaggle_submission(
        results=results,
        sample_csv_path=paths.data_dir / "submission.csv",
        output_csv_path=paths.output_path,
        category_predictions=category_predictions,
    )


# --- End src/pipeline.py ---


In [6]:
paths, config = bootstrap(output_filename="solutions_SeaFour.csv")
frames = load_project_frames(paths, config=config)
ground_truth = load_ground_truth(paths.data_dir / "qgts_train.json")

print(f"Runtime environment: {paths.runtime_env}")
print(f"Project directory  : {paths.project_dir}")
print(f"Data directory     : {paths.data_dir}")
print(f"Cache directory    : {paths.cache_dir}")
print(f"Documents          : {len(frames.docs):,}")
print(f"Train queries      : {len(frames.train_queries):,}")
print(f"Test queries       : {len(frames.test_queries):,}")
print(f"Final model        : {config.retrieval_pipeline.final_model}")
print(f"Submit top_k       : {config.retrieval_pipeline.submit_top_k:,}")

prepared_retrievers = prepare_retrievers(frames, paths, config=config)
print(f"Prepared retrievers: {sorted(prepared_retrievers.keys())}")

category_artifacts = predict_categories(frames, paths, ground_truth=ground_truth, config=config)
if category_artifacts.classifier_artifacts is None:
    print("Category filtering disabled in config.")
else:
    print(f"Train category predictions: {len(category_artifacts.train_query_category_map):,}")
    print(f"Test category predictions : {len(category_artifacts.test_query_category_map):,}")
    print(f"Category accuracy         : {category_artifacts.classifier_accuracy:.5f}")

cross_encoder_reranker = build_cross_encoder_reranker(frames, paths, ground_truth, config=config)
if cross_encoder_reranker is None:
    print("Cross-encoder reranking disabled in config.")
else:
    print("Cross-encoder reranker is ready.")

test_results, test_category_predictions = run_first_stage_retrieval(
    frames=frames,
    paths=paths,
    prepared_retrievers=prepared_retrievers,
    category_artifacts=category_artifacts,
    split="test",
    config=config,
)
print(f"First-stage results: {len(test_results):,} queries")

test_results = rerank_retrieval_results(
    results=test_results,
    frames=frames,
    category_artifacts=category_artifacts,
    cross_encoder_reranker=cross_encoder_reranker,
    split="test",
    config=config,
)
print("Reranking step completed.")

write_submission(test_results, paths, category_predictions=test_category_predictions)
print(f"Submission written to: {paths.output_path}")

submission_preview = pd.read_csv(paths.output_path)
submission_preview.head()


Runtime environment: local
Project directory  : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project
Data directory     : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/data
Cache directory    : /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/cache
Documents          : 216,041
Train queries      : 327
Test queries       : 141
Final model        : embedding
Submit top_k       : 12,500
Loading model weights from cache: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/cache/sentence_transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading docs embeddings from cache: docs_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_6485efc872cab480.npy
Prepared retrievers: ['embedding']
Loading category classifier from cache: category_classifier_e3f41fe8a612f7c0.pkl
Train category predictions: 327
Test category predictions : 141
Category accuracy         : 0.92661
Loading cross-encoder from cache: cross-encoder_ms-marco-MiniLM-L6-v2_c2a686d8799ae524


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Cross-encoder reranker is ready.
Loading queries_android_filtered embeddings from cache: queries_android_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_0ccd74b428f20fd2.npy
Loading queries_gaming_filtered embeddings from cache: queries_gaming_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_348840ac46238ca8.npy
Loading queries_unix_filtered embeddings from cache: queries_unix_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_c5bf962af271cb3a.npy
Loading queries_tex_filtered embeddings from cache: queries_tex_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_ee11070e1d0fd1d2.npy
Loading queries_programmers_filtered embeddings from cache: queries_programmers_filtered_all-MiniLM-L6-v2_14a6ec03f4bc4bf4_a49002cd8d58734b.npy
First-stage results: 141 queries
  [CrossEncoder] reranked 141/141 queries (top_m=45, category_bonus=0.50, total_pairs=6,345)
Reranking step completed.
Submission written to: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/solutions_SeaFour.csv


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""74a05d0b-2e0f-487d-b426-8bb303e78375_109602""...",programmers
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""3fd76a85-c69c-47de-b4b6-5dedf515b0a2_30246"",...",unix
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",android
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""a9963b2e-a02c-42fc-8fd8-2aa428ed1579_29888"",...",unix
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""59108dd2-5f5a-4add-b947-3303c8aaa891_123331""...",tex
